In [ ]:
import numpy as np
import subprocess
import os

# ============================================================
# OPTIONAL: SET NUMBER OF CPU THREADS
# ============================================================

os.environ["OMP_NUM_THREADS"] = "4"

# ============================================================
# REFERENCE CENTRE
# ============================================================

ra0  = 180.1649901
dec0 = -1.449266441

# ============================================================
# CONVERT RA/DEC → ARCSEC
# ============================================================

def sky_to_arcsec(ra, dec):

    x = (ra - ra0) * 3600 * np.cos(np.deg2rad(dec0))
    y = (dec - dec0) * 3600

    return x, y

# ============================================================
# CLUSTER 1 GALAXIES
# ============================================================

cluster1 = [

(180.1711571,-1.448282774,0.2712179014),
(180.1706384,-1.45269478,0.2681773725),
(180.1670402,-1.452993158,0.2676518659),
(180.1669367,-1.451199272,0.2679437871),
(180.1649901,-1.449266441,0.2650858553)

]

# ============================================================
# CLUSTER 2 GALAXIES
# ============================================================

cluster2 = [

(180.1666591,-1.444698257,0.3280670446),
(180.1646906,-1.458333486,0.3256229065),
(180.1642845,-1.451220042,0.3284340538),
(180.1622249,-1.452360391,0.327088849),
(180.1617416,-1.459161248,0.3267833107)

]

# ============================================================
# MULTIPLE IMAGE SYSTEMS
# ============================================================

images = [

("1a",180.1539267,-1.458720556,1.419649495,1),
("1b",180.1533667,-1.458059722,1.419936636,1),
("1c",180.1655904,-1.449691111,1.41905658,1),

("2a",180.1554108,-1.460539167,1.406909888,2),
("2b",180.1549083,-1.452816667,1.405739763,2),
("2c",180.1653792,-1.447126389,1.406608809,2)

]

# ============================================================
# WRITE IMAGE FILE
# ============================================================

with open("images.dat", "w") as f:

    for name, ra, dec, z, system_id in images:

        x, y = sky_to_arcsec(ra, dec)

        f.write(
            f"{system_id} {name} "
            f"{x:.4f} {y:.4f} {z:.5f}\n"
        )

print("Created images.dat")

# ============================================================
# SIMPLE FIXED HALO BLOCK
# ============================================================

def halo_block(
    x,
    y,
    z,
    sigma,
    core,
    cut
):

    return f"""

potential
    profil 81

    x_centre {x:.4f}
    y_centre {y:.4f}

    z_lens {z:.5f}

    v_disp {sigma}

    core_radius {core}
    cut_radius {cut}

    ellipticite 0.0
    angle_pos 0.0

end

"""

# ============================================================
# BEGIN PARAMETER FILE
# ============================================================

par = """

runmode
    inverse 3
end


grille
    nlens 32
end


cosmology
    H0 73.5
    omegaM 0.3
    omegaX 0.7
    wX -1
end


champ
    xmin -80
    xmax 80
    ymin -80
    ymax 80
end


image
    multfile images.dat
end

"""

# ============================================================
# MAIN CLUSTER HALO
# ============================================================

par += halo_block(
    x=0.0,
    y=0.0,
    z=0.27,
    sigma=1000,
    core=6,
    cut=250
)

# ============================================================
# SECOND CLUSTER HALO
# ============================================================

x2_main, y2_main = sky_to_arcsec(
    180.1666591,
    -1.444698257
)

par += halo_block(
    x=x2_main,
    y=y2_main,
    z=0.327,
    sigma=700,
    core=5,
    cut=180
)

# ============================================================
# CLUSTER 1 GALAXIES
# ============================================================

for ra, dec, z in cluster1:

    x, y = sky_to_arcsec(ra, dec)

    par += halo_block(
        x=x,
        y=y,
        z=z,
        sigma=250,
        core=0.15,
        cut=40
    )

# ============================================================
# CLUSTER 2 GALAXIES
# ============================================================

for ra, dec, z in cluster2:

    x, y = sky_to_arcsec(ra, dec)

    par += halo_block(
        x=x,
        y=y,
        z=z,
        sigma=220,
        core=0.15,
        cut=35
    )

# ============================================================
# FINISH PARAM FILE
# ============================================================

par += """

fini

"""

# ============================================================
# WRITE PARAM FILE
# ============================================================

with open("model.par", "w") as f:
    f.write(par)

print("Created model.par")

# ============================================================
# SHOW PARAM FILE
# ============================================================

print("\n========== model.par ==========\n")
print(par)

# ============================================================
# RUN LENSTOOL
# ============================================================

print("\nRunning Lenstool...\n")

try:

    result = subprocess.run(
        ["lenstool", "model.par"],
        capture_output=True,
        text=True
    )

    print("\n========== LENSTOOL STDOUT ==========\n")
    print(result.stdout)

    print("\n========== LENSTOOL STDERR ==========\n")
    print(result.stderr)

    print("\nReturn code:", result.returncode)

except FileNotFoundError:

    print("\nERROR: Lenstool executable not found.")

except Exception as e:

    print("\nERROR:")
    print(e)

Created images.dat
Created model.par

========== model.par ==========



runmode
    inverse 3
end


grille
    nlens 32
end


cosmology
    H0 73.5
    omegaM 0.3
    omegaX 0.7
    wX -1
end


champ
    xmin -80
    xmax 80
    ymin -80
    ymax 80
end


image
    multfile images.dat
end



potential
    profil 81

    x_centre 0.0000
    y_centre 0.0000

    z_lens 0.27000

    v_disp 1000

    core_radius 6
    cut_radius 250

    ellipticite 0.0
    angle_pos 0.0

end



potential
    profil 81

    x_centre 6.0065
    y_centre 16.4455

    z_lens 0.32700

    v_disp 700

    core_radius 5
    cut_radius 180

    ellipticite 0.0
    angle_pos 0.0

end



potential
    profil 81

    x_centre 22.1941
    y_centre 3.5412

    z_lens 0.27122

    v_disp 250

    core_radius 0.15
    cut_radius 40

    ellipticite 0.0
    angle_pos 0.0

end



potential
    profil 81

    x_centre 20.3274
    y_centre -12.3420

    z_lens 0.26818

    v_disp 250

    core_radius 0.15
    cut_radius 40